# Johansen Cointegration Test + Conditional VECM (Issue #47)
**DAMO-699 Capstone Project, Group 5**

## 1. Introduction & Motivation

Both VAR baselines fail to beat the naïve Random Walk benchmark at any forecast horizon:
- **VAR-AIC** (lag=10, issue #28): naïve significantly better at h=1 on MAE (p=0.0005)
- **VAR-BIC** (lag=0, issue #29): reduces to a drift model; naïve significantly better at h=1 on MAE (p=0.0124)

Pure differencing discards any long-run equilibrium information. If the level series share cointegrating relationships, the Error Correction Term (ECT) in a VECM may recover predictive signal that the differenced VAR cannot access.

> **Decision Justification:** This analysis is the next logical step in the modeling pipeline per the proposal §5.2 commitment and the dependency chain from `SPRINT_AUG21_RECOVERY.md`. The Johansen test screens for equilibrium relationships; only if they exist (r ≥ 1) is VECM estimation warranted.

In [ ]:
import sys
from pathlib import Path
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

warnings.filterwarnings("ignore", category=Warning, module="statsmodels")

# Add project root to path
PROJECT_ROOT = Path.cwd().resolve().parents[1]
sys.path.insert(0, str(PROJECT_ROOT))
sys.path.insert(0, str(PROJECT_ROOT / "src"))

from src.johansen_vecm import (
    load_gold_levels, confirm_i1, select_lag_levels,
    run_johansen_all_specs, fit_and_summarize_vecm,
    evaluate_vecm, dm_report_vecm,
    FEATURE_SET_5VAR, FEATURE_SET_6VAR, JOHANSEN_SPECS,
    TARGET, HORIZONS, MAX_LAG_SEARCH, MIN_TRAIN, STEP, ALPHA,
)
from statsmodels.tsa.api import VAR

print("Setup complete.")

## 2. Variable Selection

### Primary System: 5 Variables
`yield_spread_10y_2y` (target), `overnight_rate`, `us_treasury_10y`, `fed_funds_rate`, `cpi_yoy`

> **Decision Justification:** The same 5-variable system used by both VAR baselines (issues #28, #29). This ensures direct comparability — the VECM and VARs operate on the identical information set, so any performance difference is attributable to the error correction mechanism, not to variable selection. Additionally, Johansen tests lose power rapidly as system dimension grows relative to sample size; with n≈4,268, a 5-variable system preserves adequate statistical power.

### Robustness Check: 6 Variables (+ usdcad)

> **Decision Justification:** The USD/CAD exchange rate is the key international transmission channel for Canadian yields given the tight Canada-US economic linkage. Adding it tests whether cross-border dynamics introduce cointegrating relationships absent in the domestic-only system.

## 3. Data Loading & Input Series

> **Decision Justification — Raw Level Series:** The Johansen (1991) procedure is specifically designed to detect long-run equilibrium relationships among **non-stationary I(1) variables in levels**. Applying it to already-differenced I(0) series would test for cointegration among stationary variables, which is not meaningful. The EDA notebook (`01_eda`) confirmed all 5 series are I(1).

> **Decision Justification — Gold-layer CSV:** Loading from the canonical `data/processed/gold_features.csv` ensures a single validated source with zero nulls, release-date-aligned CPI (no lookahead bias), and cross-border holiday alignment (outer join with ffill limit=2).

In [ ]:
levels = load_gold_levels(FEATURE_SET_5VAR)
print(f"Shape: {levels.shape}")
print(f"Date range: {levels.index.min().date()} to {levels.index.max().date()}")
levels.describe()

## 4. Stationarity Confirmation (ADF Tests)

> **Decision Justification:** ADF confirmation is a prerequisite for Johansen testing. The procedure assumes all variables are I(1): including an I(0) variable would inflate the estimated cointegration rank, while including an I(2) variable violates the test's asymptotic theory. We verify both conditions: non-stationarity in levels (p ≥ 0.05) AND stationarity in first differences (p < 0.05).

In [ ]:
adf_results = confirm_i1(levels)
adf_results

## 5. Lag Order Selection (Levels VAR)

> **Decision Justification — Independent Lag Selection:** The VAR baseline (issue #28) selected lag p=10 by AIC on the **differenced** system. However, the Johansen test operates on a VAR in **levels**, where p lags in levels correspond to p−1 lags in the VECM representation (Lütkepohl, 2005). The optimal lag order can differ between levels and differences, so we perform independent AIC/BIC/HQIC selection on the level series up to maxlags=15.

In [ ]:
aic_lag_levels, order_results = select_lag_levels(levels)
k_ar_diff = max(1, aic_lag_levels - 1)

# Also get differenced VAR lag for comparison
diffed = levels.diff().dropna()
diffed.columns = [f"d_{c}" for c in levels.columns]
aic_lag_diff = VAR(diffed).select_order(maxlags=MAX_LAG_SEARCH).aic
aic_lag_diff = max(1, aic_lag_diff)

print(f"\nLevels VAR AIC lag: {aic_lag_levels} → VECM k_ar_diff = {k_ar_diff}")
print(f"Differenced VAR AIC lag: {aic_lag_diff} (for comparison model)")

## 6. Johansen Cointegration Test

> **Decision Justification — Three Deterministic Specifications:** We test all three deterministic cases supported by `statsmodels.coint_johansen` (det_order ∈ {-1, 0, 1}), corresponding to: (1) no deterministic terms, (2) restricted constant in the cointegrating equation, (3) restricted constant + restricted linear trend. This provides academic rigor (Pantula principle) while avoiding cherry-picking a single specification.

> **Decision Justification — Trace Statistic as Primary Criterion:** The trace test has better small-sample power properties than the maximum eigenvalue test (Lütkepohl, Saikkonen & Trenkler, 2001) and is the more commonly used in applied macro-finance. The max eigenvalue statistic is reported for transparency but the trace statistic governs the rank decision. The **restricted constant** (det_order=0) is the primary specification — interest rates and yield spreads exhibit mean reversion toward a shifting equilibrium driven by monetary policy cycles.

In [ ]:
johansen_df, primary_rank = run_johansen_all_specs(levels, k_ar_diff)
print(f"\n{'='*50}")
print(f"PRIMARY RESULT: Cointegration rank r = {primary_rank}")
print(f"{'='*50}")

In [ ]:
# Eigenvalue bar chart for primary specification (det_order=0)
primary = johansen_df[johansen_df["det_order"] == 0]

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Trace statistics vs critical values
ax = axes[0]
r_nulls = primary["r_null"].values
ax.bar(r_nulls - 0.15, primary["trace_stat"], width=0.3, label="Trace Statistic", color="steelblue")
ax.bar(r_nulls + 0.15, primary["trace_cv_95"], width=0.3, label="95% Critical Value", color="coral", alpha=0.7)
ax.set_xlabel("H₀: r ≤")
ax.set_ylabel("Statistic")
ax.set_title("Trace Test (Restricted Constant)")
ax.set_xticks(r_nulls)
ax.legend()

# Eigenvalues
ax = axes[1]
ax.bar(range(len(primary)), primary["eigenvalue"].values, color="steelblue")
ax.set_xlabel("Component")
ax.set_ylabel("Eigenvalue")
ax.set_title("Eigenvalues (Restricted Constant)")
ax.set_xticks(range(len(primary)))

plt.tight_layout()
plt.show()

## 7. VECM Estimation

This section executes **conditionally**: only if the Johansen test found cointegrating vectors (r ≥ 1) under the primary specification.

If r ≥ 1, we estimate a VECM and extract:
- **Cointegrating vectors (β):** The long-run equilibrium relationships
- **Adjustment coefficients (α):** The speed at which each variable corrects deviations from equilibrium
- **Error Correction Terms (ECT):** The time series of deviations from long-run equilibrium

In [ ]:
if primary_rank >= 1:
    det_spec = JOHANSEN_SPECS[0]["vecm_det"]
    vecm_fit = fit_and_summarize_vecm(levels, k_ar_diff, primary_rank, det_spec)
else:
    print("r = 0: No cointegrating vectors found. VECM not estimated.")
    print("\nThis is a SUBSTANTIVE FINDING — see Section 10 for interpretation.")

## 8. Forecast Evaluation

> **Decision Justification — Same Expanding-Window Protocol:** We use MIN_TRAIN=500, STEP=5, horizons h ∈ {1, 5, 20} trading days, with evaluation in levels (not differences) per proposal §5.4. This ensures apples-to-apples comparability with both VAR baselines.

> **Decision Justification — 4-Way DM Comparison:** Testing VECM against (1) naïve random walk (absolute benchmark), (2) VAR-AIC (best-fit differenced model, lag=10), and (3) VAR-BIC (most parsimonious differenced model, lag=0/drift) isolates whether the Error Correction Term adds predictive value above and beyond each alternative. The Diebold-Mariano test uses Harvey correction and Bartlett-kernel long-run variance estimation, matching the existing baselines.

In [ ]:
if primary_rank >= 1:
    det_spec = JOHANSEN_SPECS[0]["vecm_det"]
    eval_metrics, eval_raw = evaluate_vecm(
        levels, k_ar_diff, primary_rank, det_spec, aic_lag_diff
    )
    display(eval_metrics)
else:
    print("Skipped: no VECM to evaluate (r = 0).")

In [ ]:
if primary_rank >= 1:
    dm_results = dm_report_vecm(eval_raw)
    display(dm_results)
else:
    print("Skipped: no DM tests (r = 0).")

## 9. Robustness Check: 6-Variable System (+ usdcad)

> **Decision Justification:** Adding `usdcad` tests whether cross-border exchange rate dynamics introduce cointegrating relationships absent in the 5-variable domestic system. The USD/CAD exchange rate is the primary international transmission channel for Canadian interest rates.

In [ ]:
print("Loading 6-variable system...")
levels_6 = load_gold_levels(FEATURE_SET_6VAR)
print(f"Shape: {levels_6.shape}")

print("\nADF tests...")
adf_6 = confirm_i1(levels_6)

print("\nLag selection...")
aic_lag_6, _ = select_lag_levels(levels_6)
k_ar_diff_6 = max(1, aic_lag_6 - 1)

print("\nJohansen test...")
johansen_6, rank_6 = run_johansen_all_specs(levels_6, k_ar_diff_6)
print(f"\n6-var cointegration rank: r = {rank_6}")

if rank_6 >= 1:
    diffed_6 = levels_6.diff().dropna()
    diffed_6.columns = [f"d_{c}" for c in levels_6.columns]
    aic_diff_6 = max(1, VAR(diffed_6).select_order(maxlags=MAX_LAG_SEARCH).aic)
    det_6 = JOHANSEN_SPECS[0]["vecm_det"]
    vecm_6 = fit_and_summarize_vecm(levels_6, k_ar_diff_6, rank_6, det_6)
    metrics_6, raw_6 = evaluate_vecm(levels_6, k_ar_diff_6, rank_6, det_6, aic_diff_6)
    display(metrics_6)
    dm_6 = dm_report_vecm(raw_6)
    display(dm_6)
else:
    print("r = 0 in 6-var system as well — no VECM estimated.")

## 10. Negative Result Documentation (if r = 0)

> **Decision Justification:** A negative result (no cointegration) is reported as a **substantive finding**, not an open item. Per the issue description: *"If no cointegration is found, document that explicitly — a negative result here is still a result, not an open item."*
>
> If the 5 core macro-financial variables do not share a long-run equilibrium, this is economically informative: the yield spread may not revert to a stable relationship with these fundamentals over the 2010–2026 sample. Possible explanations include:
> - **Structural breaks:** Quantitative Easing (2010–2014), COVID shock (2020), aggressive rate hiking cycle (2022–2023)
> - **Regime changes:** The Bank of Canada's forward guidance and unconventional monetary policy tools may have altered the yield curve's equilibrium dynamics
> - **Genuine absence of cointegration:** The variables may be driven by independent non-stationary processes without a common stochastic trend
>
> This result also has implications for the downstream modeling pipeline: if VECM adds no value, the focus shifts to non-linear models (LSTM, issue #49) and the Diebold-Mariano comparison across all models (issue #50).

## 11. Conclusion

This notebook implemented the Johansen cointegration test and conditional VECM estimation as committed in the project proposal (§5.2) and tracked in issue #47.

**Key findings are documented in the output CSVs:**
- `outputs/johansen_cointegration_results.csv` — Full Johansen test results
- `outputs/johansen_lag_selection.csv` — Lag selection results
- `outputs/vecm_rmse_mae_vs_naive.csv` — VECM forecast metrics (if estimated)
- `outputs/vecm_diebold_mariano.csv` — DM significance tests (if estimated)

**Next steps in the dependency chain:**
- IRF / FEVD / Granger causality (issue #48)
- ARIMA baseline
- LSTM + SHAP (issue #49)
- Diebold-Mariano across all models (issue #50)